# Global Pollution Analysis and Energy Recovery
Complete EDA, Feature Engineering, Linear Regression, Logistic Regression, and Reporting.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

df = pd.read_csv("Global_Pollution_Analysis.csv")
df.head()


## Step 1 - Data Preprocessing

In [ ]:

print("Shape:", df.shape)
print(df.info())
print(df.isnull().sum())

# Handle missing values
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Label Encoding Country
le = LabelEncoder()
df["Country_Encoded"] = le.fit_transform(df["Country"])

# Scale pollution indices
pollution_cols = [
    "Air_Pollution_Index",
    "Water_Pollution_Index",
    "Soil_Pollution_Index"
]

scaler = StandardScaler()
df[[c+"_Scaled" for c in pollution_cols]] = scaler.fit_transform(df[pollution_cols])

df.head()


## Step 2 - Exploratory Data Analysis

In [ ]:

df.describe()


In [ ]:

plt.figure(figsize=(10,6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:

yearly = df.groupby("Year")[["Air_Pollution_Index","Energy_Recovered (in GWh)"]].mean()

yearly.plot(figsize=(10,5))
plt.title("Pollution and Energy Recovery Trends")
plt.show()


In [ ]:

plt.figure(figsize=(12,5))
sns.boxplot(data=df[["Air_Pollution_Index","Water_Pollution_Index","Soil_Pollution_Index"]])
plt.title("Pollution Index Distribution")
plt.show()


## Step 3 - Feature Engineering

In [ ]:

df["Total_Energy_Consumption"] = (
    df["Energy_Consumption_Per_Capita (in MWh)"] *
    df["Population (in millions)"]
)

df["Pollution_Average"] = (
    df["Air_Pollution_Index"] +
    df["Water_Pollution_Index"] +
    df["Soil_Pollution_Index"]
)/3

df[["Total_Energy_Consumption","Pollution_Average"]].head()


## Step 4 - Linear Regression: Predict Energy Recovery

In [ ]:

features = [
    "Air_Pollution_Index",
    "Water_Pollution_Index",
    "Soil_Pollution_Index",
    "Industrial_Waste (in tons)",
    "CO2_Emissions (in MT)",
    "Renewable_Energy (%)",
    "Country_Encoded",
    "Year"
]

X = df[features]
y = df["Energy_Recovered (in GWh)"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)

pred = lr.predict(X_test)

print("R2:", r2_score(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))


## Step 5 - Logistic Regression: Pollution Severity Classification

In [ ]:

df["Pollution_Severity"] = pd.qcut(
    df["Pollution_Average"],
    q=3,
    labels=["Low","Medium","High"]
)

severity_encoder = LabelEncoder()
y_class = severity_encoder.fit_transform(df["Pollution_Severity"])

X_class = df[[
    "Air_Pollution_Index",
    "CO2_Emissions (in MT)",
    "Industrial_Waste (in tons)",
    "Water_Pollution_Index",
    "Soil_Pollution_Index"
]]

X_train, X_test, y_train, y_test = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="weighted"))
print("Recall:", recall_score(y_test, y_pred, average="weighted"))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted"))

print(classification_report(y_test, y_pred))


In [ ]:

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()



## Step 6 - Model Evaluation and Comparison

- Linear Regression: Evaluate with R², MSE, MAE.
- Logistic Regression: Evaluate with Accuracy, Precision, Recall, F1-Score.
- Compare prediction quality and classification capability.



## Step 7 - Actionable Insights

1. Higher pollution levels often correlate with industrial activity and waste generation.
2. Energy recovery can be improved through waste-to-energy technologies.
3. Countries with high industrial waste and low renewable energy should prioritize recycling and recovery systems.
4. Increased renewable energy adoption can reduce pollution intensity.
5. Continuous monitoring of pollution indicators helps improve energy recovery planning.
